In [15]:
import pandas as pd
import numpy as np
import difflib
from nltk.metrics.distance import *
from collections import defaultdict
import re

Creating table of all KRAS alterations with hamming distances to be used for calculating probabilities.

In [16]:
def hamming_distance(s1, s2):
    # Handles case when strings aren't same length
    # Returns number of character changes to transform s1 to s1 (no rearrangements allowed)
    str_len = min([len(s1), len(s2)])
    extra_dist = max([len(s1), len(s2)]) - str_len
    
    return sum([s1[i] != s2[i] for i in range(str_len)]) + extra_dist

def calculate_complement(seq):
    complement_dict = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
    return ''.join([complement_dict[i] for i in seq])

def dns_change(s1, s2):
    # Returns True/False for if s1 can be transformed to s2 via an adjacent change of length 2
    # Assumes s1 and s2 are the same length
    if (hamming_distance(s1, s2) == 2) and (len(s1) == 2) and (len(s2) == 2):
        return True
    else:
        if hamming_distance(s1, s2) == 2:
            return dns_change(s1[:-1], s2[:-1]) or dns_change(s1[1:], s2[1:])
        return False
    
def get_complementary_trinucleotide_context(data):
    # get the complementary trinucleotide context based on complementarity ad 5'->3' reading orientation
    def fix_context(ctx):
        match = re.match(r'([ACGT]?)\[([ACGT])>([ACGT])\]([ACGT]?)', ctx)
        if not match: return ctx
        left, ref, alt, right = match.groups()
        if not left: left = 'N'
        if not right: right = 'N'
        return f"{left}[{ref}>{alt}]{right}"
    
    data['Type'] = data['Type'].apply(fix_context)
    
    # Generate complementary mutations
    compl = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C', 'N': 'N'}
    
    def get_complement(ctx):
        match = re.match(r'([ACGTN])\[([ACGT])>([ACGT])\]([ACGTN])', ctx)
        if not match: return None
        left, ref, alt, right = match.groups()
        return f"{compl[right]}[{compl[ref]}>{compl[alt]}]{compl[left]}"
    
    df_compl = data.copy()
    df_compl['Type'] = data['Type'].apply(get_complement)
    df_compl['IsComplement'] = True
    data['IsComplement'] = False
    
    result_df = pd.concat([data, df_compl], ignore_index=True)
    
    return result_df

In [17]:
delta = pd.read_csv('delta_nt.csv')
delta['Change'] = delta['Wt_codon'] + ':' + delta['Mut_codon']
delta['Edit_Distance'] = delta.apply(lambda x: hamming_distance(x.Wt_codon, x.Mut_codon), axis=1)
delta['Is_DNS'] = delta.apply(lambda x: dns_change(x.Wt_codon, x.Mut_codon), axis=1)
delta=delta[delta['Edit_Distance']==1] # to only consider mutation than can be  achieved via single nucleotide substitution (remove DNS with non-adjacent substitutions)

delta

,Index,Position,Wt_aa,Wt_codon,Mut_codon,Mut_aa,Mutation,Change,Edit_Distance,Is_DNS
0,1,2,T,ACT,GCT,A,T2A,ACT:GCT,1,False
18,19,2,T,ACT,ATT,I,T2I,ACT:ATT,1,False
30,31,2,T,ACT,AAT,N,T2N,ACT:AAT,1,False
32,33,2,T,ACT,CCT,P,T2P,ACT:CCT,1,False
44,45,2,T,ACT,TCT,S,T2S,ACT:TCT,1,False
...,...,...,...,...,...,...,...,...,...,...
11928,11929,188,M,ATG,TTG,L,M188L,ATG:TTG,1,False
11932,11933,188,M,ATG,CTG,L,M188L,ATG:CTG,1,False
11943,11944,188,M,ATG,AGG,R,M188R,ATG:AGG,1,False
11957,11958,188,M,ATG,ACG,T,M188T,ATG:ACG,1,False


Creating single base substitution probability table with keys matching format of sequence slices.

______

In [24]:
sbs=pd.read_csv('SBS_GRCh38_Cosmic_v3.3.csv')
sbs=get_complementary_trinucleotide_context(sbs)
sbs['From'] = sbs['Type'].apply(lambda x: x[0] + x[2] + x[6])
sbs['To'] = sbs['Type'].apply(lambda x: x[0] + x[4] + x[6])
sbs['SBS'] = sbs['From'] + ':' + sbs['To']
del sbs['Type']

sbs = sbs.loc[:, ['SBS'] + list(sbs.columns.values[:-3])].set_index('SBS').drop('IsComplement', axis=1)
sbs

,SBS1,SBS2,SBS3,SBS4,SBS5,SBS6,SBS7a,SBS7b,SBS7c,SBS7d,...,SBS85,SBS86,SBS87,SBS88,SBS89,SBS90,SBS91,SBS92,SBS93,SBS94
SBS,,,,,,,,,,,,,,,,,,,,,
ACA:AAA,8.760230e-04,5.790000e-07,0.020920,0.042451,0.012052,0.000425,6.720000e-05,0.002344,0.004841,0.000040,...,0.006108,0.002968,0.008946,1.000000e-18,0.032297,0.002222,0.002934,0.011396,0.011628,0.015677
ACC:AAC,2.220120e-03,1.455050e-04,0.016343,0.032990,0.009337,0.000516,1.767300e-04,0.000457,0.001135,0.000754,...,0.000871,0.003735,0.004490,1.000000e-18,0.017495,0.000704,0.052013,0.009653,0.008011,0.024523
ACG:AAG,1.797270e-04,5.360000e-05,0.001808,0.016116,0.001908,0.000053,7.330000e-05,0.000192,0.000388,0.000257,...,0.000316,0.000398,0.006357,1.000000e-18,0.009971,0.000144,0.000209,0.004851,0.001817,0.001627
ACT:AAT,1.265053e-03,9.760000e-05,0.012265,0.029663,0.006636,0.000180,2.485220e-04,0.000714,0.001964,0.004051,...,0.002728,0.003639,0.004941,1.737757e-03,0.020818,0.001771,0.000130,0.007800,0.008457,0.011141
ACA:AGA,1.839055e-03,2.230000e-16,0.019813,0.006931,0.010144,0.000471,6.510000e-05,0.000009,0.001123,0.001181,...,0.007268,0.052763,0.007843,1.000000e-18,0.014876,0.000513,0.000242,0.003074,0.008898,0.007048
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AAA:AGA,4.274201e-03,3.570000e-05,0.013957,0.000639,0.018550,0.001738,8.127150e-04,0.002983,0.088155,0.043061,...,0.094326,0.003450,0.006558,9.661683e-02,0.006806,0.004030,0.001267,0.014347,0.014123,0.016843
TAA:TCA,2.170000e-16,1.640000e-05,0.007161,0.000372,0.005149,0.000103,1.260260e-04,0.000944,0.018998,0.000207,...,0.007011,0.007041,0.008034,3.016341e-02,0.006769,0.018419,0.005828,0.002182,0.052961,0.004024
GAA:GCA,5.520000e-05,7.120000e-05,0.006401,0.000177,0.006677,0.000291,1.178600e-04,0.001581,0.017674,0.000117,...,0.006520,0.002412,0.002523,2.114288e-02,0.002980,0.000189,0.000145,0.000306,0.013518,0.001260


In [25]:
# example from giacomelli et al 2017, probabilities for both trinuc context shoudl match
codon1='AGC:ATC'
codon2='GCT:GAT'

sbs.loc[lambda x: x.index.isin([codon1, codon2])]

,SBS1,SBS2,SBS3,SBS4,SBS5,SBS6,SBS7a,SBS7b,SBS7c,SBS7d,...,SBS85,SBS86,SBS87,SBS88,SBS89,SBS90,SBS91,SBS92,SBS93,SBS94
SBS,,,,,,,,,,,,,,,,,,,,,
GCT:GAT,2.190000e-16,0.000051,0.009969,0.021138,0.006789,0.00122,2.220000e-16,0.000147,0.000361,0.000073,...,0.00076,0.008336,0.001055,0.003446,0.007202,0.0004,0.001153,0.006536,0.003088,0.011314
AGC:ATC,2.190000e-16,0.000051,0.009969,0.021138,0.006789,0.00122,2.220000e-16,0.000147,0.000361,0.000073,...,0.00076,0.008336,0.001055,0.003446,0.007202,0.0004,0.001153,0.006536,0.003088,0.011314


In [26]:
#KRAS sequence. Need to add at G to the beginning (last nucleotide in start codon 'ATG') and T at the end (first nucleotide in the stop codon).

kras_sequence = ''.join(delta.loc[:, ['Position', 'Wt_codon']].drop_duplicates()['Wt_codon'].values)
delta_seq = 'G' + kras_sequence + 'T'

In [27]:
# For each nucleotide, extract its trinucleotide context.
codon_rfs = [[delta_seq[i:i+3] for i in range(j, j+3)] for j in range(0, len(delta_seq)-2, 3)]

codon_rfs

[['GAC', 'ACT', 'CTG'],
 ['TGA', 'GAA', 'AAT'],
 ['ATA', 'TAT', 'ATA'],
 ['TAA', 'AAA', 'AAC'],
 ['ACT', 'CTT', 'TTG'],
 ['TGT', 'GTG', 'TGG'],
 ['GGT', 'GTA', 'TAG'],
 ['AGT', 'GTT', 'TTG'],
 ['TGG', 'GGA', 'GAG'],
 ['AGC', 'GCT', 'CTG'],
 ['TGG', 'GGT', 'GTG'],
 ['TGG', 'GGC', 'GCG'],
 ['CGT', 'GTA', 'TAG'],
 ['AGG', 'GGC', 'GCA'],
 ['CAA', 'AAG', 'AGA'],
 ['GAG', 'AGT', 'GTG'],
 ['TGC', 'GCC', 'CCT'],
 ['CTT', 'TTG', 'TGA'],
 ['GAC', 'ACG', 'CGA'],
 ['GAT', 'ATA', 'TAC'],
 ['ACA', 'CAG', 'AGC'],
 ['GCT', 'CTA', 'TAA'],
 ['AAT', 'ATT', 'TTC'],
 ['TCA', 'CAG', 'AGA'],
 ['GAA', 'AAT', 'ATC'],
 ['TCA', 'CAT', 'ATT'],
 ['TTT', 'TTT', 'TTG'],
 ['TGT', 'GTG', 'TGG'],
 ['GGA', 'GAC', 'ACG'],
 ['CGA', 'GAA', 'AAT'],
 ['ATA', 'TAT', 'ATG'],
 ['TGA', 'GAT', 'ATC'],
 ['TCC', 'CCA', 'CAA'],
 ['AAC', 'ACA', 'CAA'],
 ['AAT', 'ATA', 'TAG'],
 ['AGA', 'GAG', 'AGG'],
 ['GGA', 'GAT', 'ATT'],
 ['TTC', 'TCC', 'CCT'],
 ['CTA', 'TAC', 'ACA'],
 ['CAG', 'AGG', 'GGA'],
 ['GAA', 'AAG', 'AGC'],
 ['GCA', 'CAA', 

##### Algorithm Ideas
- If there is only one nucleotide that needs to change to create the aa change, it is treated as before where they are just saved and summed up at the end per signature (e.g. if T2A can happen 3 ways from one nucleotide change, then the probability of those three will be summed up at the end per signature)
- If there are two nucleotides that need to change to create the aa change, the probabilities of each single nucleotide change are multiplied and then saved and then summed at the end per signature (e.g. if T2A can happen 2 ways from two nucleotide changes, the probabilities of the two single nucleotide changes for the first way are multiplied and then summed with the product of the probabilities of the two single nucleotide changes for the second way)
- These are all summed up at the amino acid change level at the end to get an overall probability for aa changing

kras_mutational_prob = defaultdict(list)

for codon_position in delta['Position'].unique():
    position_delta=delta.loc[lambda x: (x['Position'] == codon_position) &  (x['Edit_Distance'] == 1) & (~x['Is_DNS'])].copy()
    wt_codon = position_delta['Wt_codon'].values[0] # 
    position_codon_rfs = codon_rfs[codon_position-2] # Getting trinucleotid context around each nucleotide in that codon

    for mutation in position_delta['Mut_codon'].values:
        aa_change = position_delta.loc[position_delta['Mut_codon'] == mutation, 'Mutation'].values[0] # mutant amino acid generated by the mutation
        position_of_change = [j for j in range(len(wt_codon)) if wt_codon[j] != mutation[j]] # position of nucleotide change in codon

        # Case when two non-adjacent nucleotides change
        if len(position_of_change) > 1:
            dnp_mutational_prob = []
        
        for poc in position_of_change:
            mut_codon_frame = list(position_codon_rfs[poc])
            mut_codon_frame[1] = mutation[poc] # Changing middle nucleotide to mutant
            
            # COSMIC is indexed by WT Codon:Mutant Codon
            codon_change = f'{position_codon_rfs[poc]}:{''.join(mut_codon_frame)}'
            
            try:
                probability = sbs.loc[codon_change, :]
            except KeyError:
                # Complement is in sbs instead of standard
                wt_codon_comp = calculate_complement(position_codon_rfs[poc])
                mut_codon_comp = calculate_complement(''.join(mut_codon_frame))
                
                # Complements need to be reversed since strand direction is opposite
                comp_codon_change = f'{wt_codon_comp[::-1]}:{mut_codon_comp[::-1]}'
                
                probability = sbs.loc[comp_codon_change, :]
            
            # Adding probabilities to amino acid change
            if len(position_of_change) == 1:
                kras_mutational_prob[aa_change].append(probability)
            else:
                # Need to multiply probabilities if it requires two nucleotide changes (so log it so we can add them later which is same as multiplying)
                dnp_mutational_prob.append(np.log(probability))
                print('DBS')
        
        if len(position_of_change) > 1:
            kras_mutational_prob[aa_change].append(np.exp(np.sum(dnp_mutational_prob, axis=0)))

In [28]:
# --- version to remve the complement searching since i'm inputting it ahead 

kras_mutational_prob = defaultdict(list)

for codon_position in delta['Position'].unique():
    position_delta = delta.loc[lambda x: (x['Position'] == codon_position) & (x['Edit_Distance'] == 1) & (~x['Is_DNS'])].copy()
    wt_codon = position_delta['Wt_codon'].values[0]
    position_codon_rfs = codon_rfs[codon_position - 2]  # Getting trinucleotide context around each nucleotide in that codon

    for mutation in position_delta['Mut_codon'].values:
        aa_change = position_delta.loc[position_delta['Mut_codon'] == mutation, 'Mutation'].values[0]  # mutant amino acid generated by the mutation
        position_of_change = [j for j in range(len(wt_codon)) if wt_codon[j] != mutation[j]]  # position of nucleotide change in codon

        # Case when two non-adjacent nucleotides change
        if len(position_of_change) > 1:
            dnp_mutational_prob = []
            print('len > 1')
        
        for poc in position_of_change:
            mut_codon_frame = list(position_codon_rfs[poc])
            mut_codon_frame[1] = mutation[poc]  # Changing middle nucleotide to mutant
            
            # COSMIC is indexed by WT Codon:Mutant Codon
            codon_change = f"{position_codon_rfs[poc]}:{''.join(mut_codon_frame)}"
            
            probability = sbs.loc[codon_change, :]
            
            # Adding probabilities to amino acid change
            if len(position_of_change) == 1:
                kras_mutational_prob[aa_change].append(probability)
            else:
                # Need to multiply probabilities if it requires two nucleotide changes (so log it so we can add them later which is same as multiplying)
                dnp_mutational_prob.append(np.log(probability))
                print('DBS')
        
        if len(position_of_change) > 1:
            kras_mutational_prob[aa_change].append(np.exp(np.sum(dnp_mutational_prob, axis=0)))

In [29]:
mutational_probabilities = kras_mutational_prob.copy()

for aa_change in mutational_probabilities:
    mutational_probabilities[aa_change] = np.sum(kras_mutational_prob[aa_change], axis=0) # Summing probabilities per signature for cases where the mutaiton can be achieved via multiple SBS

In [30]:
mutational_probabilities = pd.DataFrame.from_dict(mutational_probabilities)
mutational_probabilities.index = sbs.columns.values
mutational_probabilities = mutational_probabilities.transpose()
mutational_probabilities

,SBS1,SBS2,SBS3,SBS4,SBS5,SBS6,SBS7a,SBS7b,SBS7c,SBS7d,...,SBS85,SBS86,SBS87,SBS88,SBS89,SBS90,SBS91,SBS92,SBS93,SBS94
T2A,1.860910e-03,0.000027,0.005619,0.001176,0.007899,1.099922e-03,0.000074,2.210000e-16,0.000420,0.002300,...,0.011549,0.006828,0.003847,3.444231e-03,0.001636,0.000092,0.000134,0.002676,0.002798,0.001135
T2I,9.468128e-03,0.001856,0.012165,0.004273,0.022086,2.129832e-02,0.007436,1.086161e-02,0.007494,0.032911,...,0.004582,0.004595,0.010666,1.000000e-18,0.029742,0.000652,0.002317,0.016541,0.006537,0.016606
T2N,1.265053e-03,0.000098,0.012265,0.029663,0.006636,1.799860e-04,0.000249,7.140510e-04,0.001964,0.004051,...,0.002728,0.003639,0.004941,1.737757e-03,0.020818,0.001771,0.000130,0.007800,0.008457,0.011141
T2P,2.180000e-16,0.000132,0.002331,0.000252,0.001701,2.110660e-04,0.000120,3.199250e-04,0.001648,0.000022,...,0.000796,0.000684,0.001223,1.722116e-03,0.001263,0.000046,0.001374,0.000293,0.002691,0.000631
T2S,1.253034e-03,0.000221,0.024945,0.007157,0.013035,7.385400e-04,0.000318,5.417690e-04,0.004187,0.003022,...,0.014074,0.092830,0.013045,9.221114e-03,0.013447,0.001101,0.000395,0.006056,0.018313,0.008651
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
M188K,1.117281e-03,0.000025,0.009610,0.012364,0.009079,2.200000e-16,0.000583,1.555801e-03,0.000210,0.003670,...,0.021017,0.001986,0.003798,5.744160e-04,0.010619,0.000354,0.000158,0.006991,0.005296,0.004019
M188L,9.523320e-04,0.000330,0.009408,0.010437,0.011186,1.228000e-04,0.002397,2.566622e-03,0.024134,0.003757,...,0.135076,0.006599,0.017638,5.570096e-02,0.016668,0.066794,0.002571,0.012102,0.035739,0.014203
M188R,2.626590e-04,0.000002,0.006281,0.001416,0.007905,1.060980e-04,0.000274,1.466043e-03,0.000600,0.010515,...,0.004466,0.001715,0.002656,2.297664e-03,0.005086,0.000231,0.001206,0.001235,0.005118,0.002072
M188T,1.038870e-04,0.000013,0.012262,0.004986,0.038029,4.204267e-03,0.000186,1.495962e-03,0.001063,0.009116,...,0.005049,0.003507,0.000962,6.318577e-03,0.014620,0.000108,0.000603,0.039993,0.006267,0.002821


In [31]:
#mutational_probabilities.to_csv('COSMIC-v3-KRAS-Mutational-Probabilities-Excluding-DNT.tsv', sep='\t')
mutational_probabilities.to_csv('COSMIC_v3_KRAS_Mutational_Probabilities_SBS_only.csv')

____ 
Below has not been used for the manuscript

#### Performing Analysis Using Adjacent DNS

In [ ]:
dbs = pd.read_csv('DBS_GRCh38_Cosmic_v3.3.csv')

dbs['From'] = dbs['Type'].apply(lambda x: x[0:2])
dbs['To'] = dbs['Type'].apply(lambda x: x[3:])
dbs['DBS'] = dbs['From'] + ':' + dbs['To']
del dbs['Type']

dbs = dbs.loc[:, ['DBS'] + list(dbs.columns.values[:-3])]
dbs.set_index('DBS', inplace=True)
dbs.head()

In [ ]:
# Change is always internal (by def) so only have 2 windows per codon (since they are next to each other)
codon_rfs_dbs = [[delta_seq[i:i+2] for i in range(j, j+2)] for j in range(1, len(delta_seq)-2, 3)]

In [ ]:
kras_mutational_prob_dbs = defaultdict(list)
not_present = []

for codon_position in delta['Position'].unique():
    
    position_delta = delta.loc[delta['Position'] == codon_position, :]
    position_delta = position_delta.loc[(position_delta['Edit_Distance'] == 2) & (position_delta['Is_DNS']), :]
    wt_codon = position_delta['Wt_codon'].values[0] # Always 1 unique value
    position_codon_rfs = codon_rfs_dbs[codon_position-2] # Getting frames in codon
    
    for mutation in position_delta['Mut_codon'].values:
        aa_change = position_delta.loc[position_delta['Mut_codon'] == mutation, 'Mutation'].values[0]    
        
        # Values in this should be consecutive
        position_of_change = [j for j in range(len(wt_codon)) if wt_codon[j] != mutation[j]]
        
        # COSMIC is indexed by WT Codon:Mutant Codon
        codon_change = f'{wt_codon[position_of_change[0]:position_of_change[1]+1]}:{mutation[position_of_change[0]:position_of_change[1]+1]}'
        
        try:
            probability = dbs.loc[codon_change, :]
        except KeyError:
                # Complement is in dbs instead of standard (also needs to be reversed since strand is reverse)
                wt_codon_comp = calculate_complement(wt_codon[position_of_change[0]:position_of_change[1]+1])
                mut_codon_comp = calculate_complement(mutation[position_of_change[0]:position_of_change[1]+1])
                
                comp_codon_change = f'{wt_codon_comp[::-1]}:{mut_codon_comp[::-1]}'
                
                probability = dbs.loc[comp_codon_change, :]
        
        if aa_change == 'T20I':
            print(probability)
        # Adding probabilities to amino acid change
        kras_mutational_prob_dbs[aa_change].append(probability)

In [ ]:
mutational_probabilities_dbs = kras_mutational_prob_dbs.copy()

for aa_change in mutational_probabilities_dbs:
    # Summing per signature
    mutational_probabilities_dbs[aa_change] = np.sum(kras_mutational_prob_dbs[aa_change], axis=0)
    
mutational_probabilities_dbs = pd.DataFrame.from_dict(mutational_probabilities_dbs)
mutational_probabilities_dbs.index = dbs.columns.values
mutational_probabilities_dbs = mutational_probabilities_dbs.transpose()
mutational_probabilities_dbs.head()

In [ ]:
#mutational_probabilities_dbs.to_csv('COSMIC-v3-KRAS-Mutational-Probabilities-DBS.tsv', sep='\t')

________________